In [ ]:
"""
filter_by_date_colab.py

Google Colab script that:
  1. Mounts Google Drive
  2. Asks the user for the Drive folder that contains the CJP CSV datasets
  3. Automatically finds all CSV files in that folder (no need to list
     filenames manually)
  4. Filters each dataset to keep only posts whose `timestamp` falls
     within 1 June 2026 to 31 July 2026 (inclusive)
  5. Saves all filtered CSVs into a zip file
  6. Downloads that zip file to your local system

Usage (in a Colab cell):
    !python filter_by_date_colab.py
or simply paste the whole script into a Colab cell and run it.

Notes on the data:
- The files are semicolon-delimited (sep=";").
- The date column is called "timestamp" and looks like:
      2026-09-01 16:46:00
- Rows with a missing/unparseable timestamp are dropped and reported.
"""

import os
import glob
import shutil
import pandas as pd

# ---- CONFIG -------------------------------------------------------------

START_DATE = "2026-06-01 00:00:00"
END_DATE = "2026-07-31 23:59:59"

DATE_COLUMN = "timestamp"
DELIMITER = ";"

OUTPUT_DIR = "/content/filtered_datasets"   # temp folder for filtered CSVs
ZIP_NAME = "filtered_datasets"              # zip file name (without .zip)

# --------------------------------------------------------------------------


def mount_drive():
    """Mount Google Drive inside Colab."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print("This script is meant to run inside Google Colab. "
              "google.colab module not found — skipping Drive mount.")


def get_input_folder():
    """Ask the user for the Drive folder containing the datasets."""
    folder = input(
        "\nEnter the full path to the Google Drive folder containing your "
        "CSV datasets\n(e.g. /content/drive/MyDrive/CJP_Datasets): "
    ).strip()

    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")

    return folder


def find_csv_files(folder):
    """Automatically find all CSV files in the given folder."""
    csv_files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {folder}")
    print(f"\nFound {len(csv_files)} CSV file(s):")
    for f in csv_files:
        print(f"  - {os.path.basename(f)}")
    return csv_files


def filter_file(input_path, output_path):
    df = pd.read_csv(input_path, sep=DELIMITER, encoding="utf-8", low_memory=False)

    if DATE_COLUMN not in df.columns:
        print(f"  [SKIP] '{DATE_COLUMN}' column not found in {input_path}")
        return

    # Parse timestamps; invalid ones become NaT
    df["_parsed_timestamp"] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")

    before = len(df)
    dropped_unparseable = df["_parsed_timestamp"].isna().sum()

    start = pd.to_datetime(START_DATE)
    end = pd.to_datetime(END_DATE)

    mask = (df["_parsed_timestamp"] >= start) & (df["_parsed_timestamp"] <= end)
    filtered = df.loc[mask].drop(columns=["_parsed_timestamp"])

    filtered.to_csv(output_path, sep=DELIMITER, index=False, encoding="utf-8")

    after = len(filtered)
    print(f"  {os.path.basename(input_path)}: {before} -> {after} rows "
          f"({dropped_unparseable} had unparseable timestamps) "
          f"saved to {output_path}")


def zip_and_download(folder_to_zip, zip_base_name):
    """Zip the filtered CSVs and trigger a download to the local system."""
    zip_path = shutil.make_archive(
        base_name=os.path.join("/content", zip_base_name),
        format="zip",
        root_dir=folder_to_zip,
    )
    print(f"\nCreated zip file: {zip_path}")

    try:
        from google.colab import files
        print("Starting download...")
        files.download(zip_path)
    except ImportError:
        print("Not running in Colab — skipping automatic download. "
              f"You can find the zip file at: {zip_path}")


def main():
    print(f"Filtering posts between {START_DATE} and {END_DATE}")

    # Step 1: Mount Google Drive
    mount_drive()

    # Step 2: Ask for the Drive folder containing the datasets
    input_folder = get_input_folder()

    # Step 3: Find all CSV files in that folder
    csv_files = find_csv_files(input_folder)

    # Step 4: Filter each dataset and save into OUTPUT_DIR
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("\nFiltering files...")
    for input_path in csv_files:
        filename = os.path.basename(input_path)
        name, ext = os.path.splitext(filename)
        output_path = os.path.join(OUTPUT_DIR, f"{name}_filtered{ext}")
        filter_file(input_path, output_path)

    # Step 5 & 6: Zip the filtered CSVs and download
    zip_and_download(OUTPUT_DIR, ZIP_NAME)


if __name__ == "__main__":
    main()

Filtering posts between 2026-06-01 00:00:00 and 2026-07-31 23:59:59
Mounted at /content/drive

Enter the full path to the Google Drive folder containing your CSV datasets
(e.g. /content/drive/MyDrive/CJP_Datasets): /content/drive/MyDrive/CSMA/Temporary Datasets

Found 5 CSV file(s):
  - 1_CJP_Protest_cleaned.csv
  - 2_CJP_JantarMantar_cleaned.csv
  - 3a_CJP_Government_Original_cleaned.csv
  - 3b_CJP_Government_AdvSearch_cleaned.csv
  - 4_CJP_NEET-UG_cleaned.csv

Filtering files...
  1_CJP_Protest_cleaned.csv: 417 -> 232 rows (0 had unparseable timestamps) saved to /content/filtered_datasets/1_CJP_Protest_cleaned_filtered.csv
  2_CJP_JantarMantar_cleaned.csv: 885 -> 338 rows (0 had unparseable timestamps) saved to /content/filtered_datasets/2_CJP_JantarMantar_cleaned_filtered.csv
  3a_CJP_Government_Original_cleaned.csv: 964 -> 45 rows (0 had unparseable timestamps) saved to /content/filtered_datasets/3a_CJP_Government_Original_cleaned_filtered.csv
  3b_CJP_Government_AdvSearch_cleaned.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
"""Preprocess and topic-filter the CJP / Jantar Mantar X datasets in Colab."""

from pathlib import Path
from urllib.parse import parse_qs, urlparse
import re
import warnings

import pandas as pd


# 1. Mount Google Drive when running in Colab.
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Google Colab not detected; using the folder path set below.")


# 2. Change this to the folder containing Group A, Group B, and Group C.
# Example: /content/drive/MyDrive/CSMA Extended Datasets
BASE_DIR = Path("/content/drive/MyDrive/CSMA/CSMA Extended Datasets ")

START = pd.Timestamp("2026-06-01 00:00:00")
END = pd.Timestamp("2026-07-31 23:59:59")
OUTPUT_DIR = BASE_DIR / "processed"


# Topic signals are checked in post text, quoted text, and hashtags.
SIGNALS = {
    "cjp": re.compile(
        r"\bcjp(?:protest|movement|party)?\b|#cjpprotest|cockroach|"
        r"कॉ.?क्रोच|कोकरोच|जनता\s*पार्टी|abhijeet|abhijit|dipke|dipte|दिपके",
        re.I,
    ),
    "jantar_mantar": re.compile(r"jantar[\s\-]*mantar|जंतर\s*मंतर", re.I),
    "neet_exam": re.compile(
        r"\bneet(?:[\s\-]*ug)?\b|\bnta\b|n\.t\.a\.|grace\s*marks|नीट|एनटीए",
        re.I,
    ),
    "paper_leak": re.compile(
        r"paper[\s\-]*leak|question\s*paper|exam\s+irregularit|पेपर\s*लीक",
        re.I,
    ),
    "protest": re.compile(
        r"protest|protester|demonstrat|dharna|andolan|आंदोलन|प्रदर्शन|धरना|march|rally|"
        r"sit[ -]?in|agitation|strike|hunger\s*strike|अनशन|धरने|join us|come to|"
        r"meet you|gathering|take to the streets|सड़कों?\s*पर\s*उतर",
        re.I,
    ),
    "government_accountability": re.compile(
        r"dharmendra\s+pradhan|education\s+minister|शिक्षा\s*मंत्री",
        re.I,
    ),
    "police": re.compile(r"police|दिल्ली पुलिस|पुलिस", re.I),
    "students": re.compile(r"student|aspirant|विद्यार्थी|छात्र|अभ्यर्थी", re.I),
}

GROUP_RE = re.compile(r"\s*group\s*([a-z])\s*\Z", re.I)
URL_RE = re.compile(r"https?://\S+", re.I)
REQUIRED_COLUMNS = [
    "collected_from_url", "id", "thread_id", "timestamp", "link", "body",
    "author", "author_fullname", "author_id", "author_bio", "author_location",
    "author_followers", "author_following", "verified", "language_guess",
    "retweet_count", "reply_count", "like_count", "quote_count", "impression_count",
    "is_retweet", "retweeted_user", "is_quote_tweet", "quote_tweet_id", "quote_author",
    "quote_body", "is_reply", "replied_author", "hashtags", "urls", "mentions",
]


def find_group_dirs(base_dir):
    found = {}
    for path in base_dir.iterdir():
        if path.is_dir():
            match = GROUP_RE.fullmatch(path.name)
            if match:
                found[match.group(1).upper()] = path
    return found


def read_export(path):
    """Read comma- or semicolon-separated exports while preserving ID strings."""
    with path.open("r", encoding="utf-8-sig", errors="replace") as stream:
        header = stream.readline()
    separator = ";" if header.count(";") > header.count(",") else ","
    malformed = {"count": 0}

    def skip_bad_row(_row):
        malformed["count"] += 1
        return None

    frame = pd.read_csv(
        path,
        sep=separator,
        engine="python",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
        encoding_errors="replace",
        on_bad_lines=skip_bad_row,
    )
    frame.columns = [str(column).strip() for column in frame.columns]
    return frame, malformed["count"]


def query_from_url(value):
    try:
        return parse_qs(urlparse(str(value)).query).get("q", [""])[0].strip()
    except Exception:
        return ""


def normalize_bool(series):
    return series.astype(str).str.strip().str.lower().isin(["yes", "true", "1"])


def normalize_handle(series):
    return (
        series.fillna("").astype(str).str.strip().str.lstrip("@").str.lower()
        .replace({"nan": "", "none": ""})
    )


def normalize_mentions(value):
    if not value:
        return ""
    parts = re.split(r"[;,\s]+", str(value))
    found, output = set(), []
    for part in parts:
        handle = part.strip().lstrip("@").lower()
        if handle and handle not in found:
            found.add(handle)
            output.append(handle)
    return ",".join(output)


def clean_text(series):
    return (
        series.fillna("").astype(str).str.replace(URL_RE, " ", regex=True)
        .str.replace(r"\s+", " ", regex=True).str.strip()
    )


def topic_labels(body, query, filename):
    text = " ".join([str(body or ""), str(query or ""), str(filename or "")])
    return [name for name, pattern in SIGNALS.items() if pattern.search(text)]


def classify_relevance(body, quote_body, hashtags, query, filename, source_url):
    """Use post content plus collection context; reject generic search results."""
    body = " ".join(str(value or "") for value in [body, quote_body, hashtags])
    query = str(query or "")
    filename = str(filename or "")
    source_url = str(source_url or "")

    from_search = bool(re.search(r"/search(?:\?|/)", source_url, re.I))
    if not from_search:
        return False, "not from an X search-results page; possible generic/home-feed record"

    labels = set(topic_labels(body, "", ""))
    cjp_in_post = "cjp" in labels
    jantar_in_post = "jantar_mantar" in labels
    exam_in_post = "neet_exam" in labels or "paper_leak" in labels
    protest_in_post = "protest" in labels
    accountability_in_post = "government_accountability" in labels

    source_context = f"{query} {filename}"
    cjp_search = bool(re.search(r"\bcjp\b|cockroach|dipke|कॉ.?क्रोच|जनता\s*पार्टी", source_context, re.I))

    if cjp_in_post:
        return True, "post text refers to CJP, its founder, or the Cockroach Janta Party"
    if cjp_search and (jantar_in_post or protest_in_post or exam_in_post):
        return True, "CJP-focused search plus protest, Jantar Mantar, or exam context in post"
    if exam_in_post and (protest_in_post or jantar_in_post):
        return True, "NEET/paper-leak topic linked to protest or Jantar Mantar in post"
    if jantar_in_post and protest_in_post and accountability_in_post:
        return True, "Jantar Mantar protest linked to the Education Minister in post"
    return False, "no clear CJP protest or NEET/paper-leak protest connection in post"


def process_file(path, group):
    frame, malformed = read_export(path)
    raw_count = len(frame)
    for column in REQUIRED_COLUMNS:
        if column not in frame.columns:
            frame[column] = ""

    frame["timestamp"] = pd.to_datetime(frame["timestamp"], errors="coerce")
    missing_id = frame["id"].astype(str).str.strip().eq("")
    invalid_timestamp = frame["timestamp"].isna()
    exceptions = frame.loc[missing_id | invalid_timestamp].copy()
    exceptions["exception_reason"] = ""
    exceptions.loc[missing_id.loc[exceptions.index], "exception_reason"] += "missing post ID; "
    exceptions.loc[invalid_timestamp.loc[exceptions.index], "exception_reason"] += "invalid/missing timestamp; "

    # Keep in-window posts even when text or author metadata is incomplete.
    frame = frame.loc[~missing_id & ~invalid_timestamp].copy()

    in_window = frame["timestamp"].between(START, END, inclusive="both")
    outside_count = int((~in_window).sum())
    frame = frame.loc[in_window].copy()

    frame["group"] = group
    frame["source_file"] = path.name
    frame["search_query"] = frame["collected_from_url"].map(query_from_url)
    frame["source_url"] = frame["collected_from_url"]
    frame["topic_evidence"] = frame.apply(
        lambda row: " ".join([row["body"], row["quote_body"], row["hashtags"]]), axis=1
    )
    frame["topic_signals"] = frame["topic_evidence"].map(
        lambda text: ",".join(topic_labels(text, "", ""))
    )
    relevance = frame.apply(
        lambda row: classify_relevance(
            row["body"], row["quote_body"], row["hashtags"], row["search_query"],
            path.name, row["collected_from_url"]
        ),
        axis=1,
    )
    frame["is_project_relevant"] = relevance.map(lambda result: result[0])
    frame["relevance_reason"] = relevance.map(lambda result: result[1])
    frame["content_eligible"] = frame["body"].astype(str).str.strip().ne("")
    frame["network_eligible"] = (
        frame["author_id"].astype(str).str.strip().ne("")
        | frame["author"].astype(str).str.strip().ne("")
    )

    frame["group"] = group
    frame["source_file"] = path.name
    excluded = frame.loc[~frame["is_project_relevant"]].copy()
    frame = frame.loc[frame["is_project_relevant"]].copy()

    # Keep the most complete copy of each relevant post inside a file.
    frame["_completeness"] = frame[REQUIRED_COLUMNS].astype(str).ne("").sum(axis=1)
    before_dedupe = len(frame)
    frame = frame.sort_values("_completeness", ascending=False).drop_duplicates("id", keep="first")
    frame = frame.drop(columns="_completeness")

    log = {
        "group": group,
        "file": path.name,
        "raw_rows": raw_count,
        "malformed_rows_skipped_by_parser": malformed,
        "rows_missing_id_or_valid_timestamp": len(exceptions),
        "outside_study_window": outside_count,
        "duplicate_ids_removed_within_file": before_dedupe - len(frame),
        "relevant_rows_kept": len(frame),
        "off_topic_or_generic_excluded": len(excluded),
        "missing_text_kept": int((~frame["content_eligible"]).sum()),
        "missing_author_kept": int((~frame["network_eligible"]).sum()),
    }
    return frame, log, exceptions, excluded


def standardize(frame):
    frame = frame.copy()
    for column in REQUIRED_COLUMNS:
        if column not in frame.columns:
            frame[column] = ""

    numeric_columns = [
        "author_followers", "author_following", "retweet_count", "reply_count",
        "like_count", "quote_count", "impression_count",
    ]
    for column in numeric_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce").astype("Int64")

    for column in ["verified", "is_retweet", "is_quote_tweet", "is_reply"]:
        frame[column] = normalize_bool(frame[column])

    for column in ["author", "replied_author", "quote_author", "retweeted_user"]:
        frame[column] = normalize_handle(frame[column])

    frame["mentions"] = frame["mentions"].map(normalize_mentions)
    frame["hashtags"] = (
        frame["hashtags"].fillna("").astype(str).str.replace(r"[;\s]+", ",", regex=True)
        .str.lower().str.strip(",")
    )
    frame["text_clean"] = clean_text(frame["body"])
    frame["id"] = frame["id"].astype(str).str.strip()
    return frame.sort_values("timestamp").reset_index(drop=True)


def main():
    base_dir = BASE_DIR.expanduser()
    if not base_dir.is_dir():
        raise FileNotFoundError(
            f"Folder not found: {base_dir}\n"
            "Edit BASE_DIR so it points to the folder containing Group A, Group B, and Group C."
        )

    group_dirs = find_group_dirs(base_dir)
    if not group_dirs:
        raise FileNotFoundError(f"No Group A/B/C folders found inside: {base_dir}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    logs, group_frames, exception_frames, excluded_frames = [], {}, [], []

    for group in ["A", "B", "C"]:
        if group not in group_dirs:
            warnings.warn(f"Group {group} folder not found; skipping it.")
            continue

        files = sorted(group_dirs[group].glob("*.csv"))
        if not files:
            warnings.warn(f"No CSV files found in {group_dirs[group]}; skipping it.")
            continue

        frames = []
        for path in files:
            frame, log, exceptions, excluded = process_file(path, group)
            frames.append(frame)
            if not exceptions.empty:
                exceptions["group"] = group
                exceptions["source_file"] = path.name
                exception_frames.append(exceptions)
            if not excluded.empty:
                excluded_frames.append(excluded)
            logs.append(log)
            print(
                f"Group {group} | {path.name}: {log['raw_rows']} raw rows, "
                f"{log['relevant_rows_kept']} relevant rows kept"
            )

        merged = pd.concat(frames, ignore_index=True)
        merged["_completeness"] = merged[REQUIRED_COLUMNS].astype(str).ne("").sum(axis=1)
        merged = merged.sort_values("_completeness", ascending=False).drop_duplicates("id", keep="first")
        merged = merged.drop(columns="_completeness")
        clean = standardize(merged)
        clean.to_csv(OUTPUT_DIR / f"Group_{group}_clean.csv", index=False, encoding="utf-8-sig")
        group_frames[group] = clean

    if not group_frames:
        raise ValueError("No group produced data. Check the folder layout and CSV files.")

    combined = pd.concat(group_frames.values(), ignore_index=True)
    combined["_completeness"] = combined[REQUIRED_COLUMNS].astype(str).ne("").sum(axis=1)
    combined = combined.sort_values("_completeness", ascending=False)

    provenance = (
        combined.groupby("id", as_index=False)
        .agg(
            groups=("group", lambda values: "|".join(sorted(set(values)))),
            source_files=("source_file", lambda values: " || ".join(sorted(set(values)))),
            search_queries=("search_query", lambda values: " || ".join(sorted(set(v for v in values if v)))),
        )
    )
    master = combined.drop_duplicates("id", keep="first").drop(columns="_completeness")
    master = master.merge(provenance, on="id", how="left", suffixes=("", "_all_sources"))
    master = master.sort_values("timestamp").reset_index(drop=True)
    master.to_csv(OUTPUT_DIR / "master_protest_dataset.csv", index=False, encoding="utf-8-sig")

    if excluded_frames:
        excluded_posts = pd.concat(excluded_frames, ignore_index=True)
        excluded_posts.to_csv(
            OUTPUT_DIR / "excluded_generic_or_off_topic_posts.csv",
            index=False,
            encoding="utf-8-sig",
        )

    report = pd.DataFrame(logs)
    report.to_csv(OUTPUT_DIR / "preprocessing_log.csv", index=False, encoding="utf-8-sig")
    if exception_frames:
        pd.concat(exception_frames, ignore_index=True).to_csv(
            OUTPUT_DIR / "data_quality_exceptions.csv", index=False, encoding="utf-8-sig"
        )

    summary = [
        f"Study window: {START} through {END}",
        f"Input CSV files processed: {len(logs)}",
        f"Relevant rows kept before cross-file deduplication: {sum(x['relevant_rows_kept'] for x in logs)}",
        f"Unique posts in master dataset: {len(master)}",
        f"Unique authors (author ID where present, otherwise handle): {master['author_id'].replace('', pd.NA).fillna(master['author']).nunique()}",
        f"Generic/off-topic in-window rows excluded before deduplication: {sum(x['off_topic_or_generic_excluded'] for x in logs)}",
        f"Posts with text usable for content analysis: {int(master['content_eligible'].sum())}",
        f"Posts with an author usable for network analysis: {int(master['network_eligible'].sum())}",
        f"Posts marked as replies: {int(master['is_reply'].sum())}",
        f"Posts marked as quotes: {int(master['is_quote_tweet'].sum())}",
        f"Posts with mentions: {int(master['mentions'].ne('').sum())}",
        "Posts without a clear topic link are excluded from the master and saved with their reason.",
        "Rows with a missing ID or invalid timestamp are listed in data_quality_exceptions.csv.",
        f"Output folder: {OUTPUT_DIR}",
    ]
    summary_text = "\n".join(summary)
    (OUTPUT_DIR / "preprocessing_summary.txt").write_text(summary_text, encoding="utf-8")
    print("\n" + summary_text)


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Group A | 1_CJP_Protest_cleaned_filtered.csv: 232 raw rows, 232 relevant rows kept
Group A | 2_CJP_JantarMantar_cleaned_filtered.csv: 338 raw rows, 335 relevant rows kept
Group A | 3rd Jantar Mantar Protestors.csv: 680 raw rows, 211 relevant rows kept
Group A | 4th कॉकरोच जनता पार्टी.csv: 220 raw rows, 220 relevant rows kept
Group B | "NEET aspirants" "Jantar Mantar".csv: 50 raw rows, 35 relevant rows kept
Group B | "paper leak" "Jantar Mantar".csv: 292 raw rows, 131 relevant rows kept
Group B | 4_CJP_NEET-UG_cleaned_filtered.csv: 145 raw rows, 136 relevant rows kept
Group C | "Jantar Mantar" government.csv: 858 raw rows, 49 relevant rows kept
Group C | 3a_CJP_Government_Original_cleaned_filtered.csv: 45 raw rows, 45 relevant rows kept
Group C | 3b_CJP_Government_AdvSearch_cleaned_filtered.csv: 42 raw rows, 42 relevant rows kept
Group C | Jantar Mantar" "Delh